# Day 02 · 고급 검색 전략 실습 (로컬)
Day 01 RAG 봇에 **하이브리드(BM25+벡터)·Reranker·HyDE·Multi-Query**를 얹어 검색 품질을 끌어올립니다. 생성·질의변환만 Qwen.


## 0 · 설치 & 설정


In [ ]:
!pip install -q sentence-transformers qdrant-client langchain-text-splitters openai rank_bm25


In [ ]:
import os
# 강사 배포: 공개 URL은 기본값에 있고, 개인 키(roster의 DLI_API_KEY)만 넣으면 됩니다.
QWEN_BASE_URL = os.getenv("QWEN_BASE_URL", "http://124.51.229.210:30001/v1")
QWEN_API_KEY  = os.getenv("QWEN_API_KEY",  os.getenv("DLI_API_KEY", "<개인-키>"))
LLM_MODEL   = "Qwen/Qwen3.5-35B-A3B"                    # 서버가 서빙하는 실제 모델 id
EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

## 1 · Day 01 검색 준비 (요약 재구성)
청크·임베딩·로컬 Qdrant까지 한 번에 세팅합니다.


In [ ]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

texts = [
    "연차 휴가는 사용 3영업일 전까지 사내포털에서 신청하고 팀장 승인을 받는다.",
    "비용 정산은 영수증 첨부 후 경영지원팀 검토, 재무팀이 최종 승인한다.",
    "재택근무는 주 2회까지 가능하며 전날 18시까지 팀 채널에 공유한다.",
    "에러코드 ERR-4021은 인증 토큰 만료를 의미하며 재로그인으로 해결한다.",
    "제품 X-200의 펌웨어 최신 버전은 2.3이며 설정 메뉴에서 업데이트한다.",
    "보안상 외부 USB는 금지이며 파일 공유는 사내 드라이브를 사용한다.",
]
embedder = SentenceTransformer(EMBED_MODEL)
def embed(xs): return embedder.encode(xs, normalize_embeddings=True)

client = QdrantClient(":memory:")
dim = embed(["x"]).shape[1]
client.recreate_collection("docs", vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
client.upsert("docs", points=[PointStruct(id=i, vector=embed([t])[0].tolist(), payload={"text": t})
                              for i, t in enumerate(texts)])

def vector_search_idx(query, n=10):
    qv = embed([query])[0]
    res = client.query_points("docs", query=qv.tolist(), limit=n).points
    return [p.id for p in res]
print("준비 완료:", len(texts), "청크")


## Lab 1 · BM25 (키워드 검색)
정확 토큰(ERR-4021, X-200)에 강한 색인 검색.


In [ ]:
import re
from rank_bm25 import BM25Okapi

# 한국어는 조사가 붙어 단순 split()이면 'ERR-4021은'·'X-200의' 같은 토큰이 되어
# 쿼리 'ERR-4021'·'X-200'과 매칭되지 않습니다(점수 0).
# 영문/숫자/하이픈 덩어리와 한글 덩어리를 분리해 '정확 토큰'을 살립니다.
def tokenize(t):
    return re.findall(r"[A-Za-z0-9\-]+|[가-힣]+", t)

tokenized = [tokenize(t) for t in texts]
bm25 = BM25Okapi(tokenized)

def bm25_search(query, k=10):
    scores = bm25.get_scores(tokenize(query))
    return sorted(range(len(scores)), key=lambda i: -scores[i])[:k]

print("BM25 'ERR-4021':", [texts[i][:20] for i in bm25_search("ERR-4021", 3)])

## Lab 2 · 하이브리드 + RRF
벡터·BM25 결과를 **순위(rank)** 로 합칩니다. k=60.


In [ ]:
def rrf(rank_lists, k=60):
    score = {}
    for ranks in rank_lists:
        for r, idx in enumerate(ranks):
            score[idx] = score.get(idx, 0) + 1/(k + r + 1)
    return sorted(score, key=lambda i: -score[i])

def hybrid(query, n=10):
    return rrf([vector_search_idx(query, n), bm25_search(query, n)])[:n]

for tag, fn in [("벡터", vector_search_idx), ("BM25", bm25_search), ("하이브리드", hybrid)]:
    print(tag, "→", [texts[i][:16] for i in fn("X-200 펌웨어 버전", 3)])


## Lab 3 · Reranker (서류전형 → 면접)
cross-encoder가 (질문, 문서)를 함께 읽어 관련성으로 다시 줄세웁니다. N=후보 → K=최종.


In [ ]:
from sentence_transformers import CrossEncoder
# 다국어 reranker (한국어 지원). 영어 전용 ms-marco-MiniLM 은 한국어에서 오작동합니다.
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")   # 로컬·다국어

def rerank(query, cand_idx, top_k=3):
    pairs = [(query, texts[i]) for i in cand_idx]
    scores = reranker.predict(pairs)
    order = sorted(range(len(cand_idx)), key=lambda j: -scores[j])
    return [cand_idx[j] for j in order[:top_k]]

q = "비용은 누가 최종 승인하나요?"
cand = hybrid(q, n=6)              # 1차: 넓게
best = rerank(q, cand, top_k=3)    # 2차: 정밀
print("rerank 전:", [texts[i][:18] for i in cand])
print("rerank 후:", [texts[i][:18] for i in best])

## Lab 4 · HyDE (가상 답변으로 검색)
짧고 모호한 질문에 '그럴듯한 가상 답'을 만들어 그걸로 검색.


In [ ]:
from openai import OpenAI
llm = OpenAI(base_url=QWEN_BASE_URL, api_key=QWEN_API_KEY)

def hyde_search(query, n=5):
    hypo = llm.chat.completions.create(model=LLM_MODEL, temperature=0.3,
        messages=[{"role":"user","content": f"다음 질문에 짧게 그럴듯한 답을 써줘:\n{query}"}]
    ).choices[0].message.content
    return vector_search_idx(hypo, n)

print(hyde_search("그거 며칠 전에 말해야 해요?"))   # 모호 질문


## Lab 5 · Multi-Query (여러 표현으로 검색)
질문을 여러 표현으로 바꿔 각각 검색 → RRF로 합침.


In [ ]:
def multi_query(query, n=6):
    prompt = f"질문을 뜻은 같되 표현이 다른 3개로 바꿔, 한 줄씩 적어줘:\n{query}"
    variants = llm.chat.completions.create(model=LLM_MODEL,
        messages=[{"role":"user","content":prompt}]).choices[0].message.content.splitlines()
    lists = [vector_search_idx(v, n) for v in variants if v.strip()]
    return rrf(lists)[:n]

print(multi_query("재택 언제까지 알려줘야 하나요?"))


## Lab 6 · 최종 파이프라인 & A/B 비교
**하이브리드(N) → reranker → Top-K → 생성**. 기준선(벡터-only)과 비교해 보세요.


In [ ]:
SYSTEM = ("아래 [자료]의 내용만 근거로 답하라. 없으면 '문서에 없음'이라고만 답하라. "
          "답 끝에 (근거)를 표기하라.")

def generate(query, idxs):
    ctx = "\n".join(f"[{i+1}] {texts[d]}" for i, d in enumerate(idxs))
    out = llm.chat.completions.create(model=LLM_MODEL, temperature=0.2,
        messages=[{"role":"system","content":SYSTEM},
                  {"role":"user","content":f"[자료]\n{ctx}\n\n[질문] {query}"}])
    return out.choices[0].message.content

def advanced_rag(query):
    cand = hybrid(query, n=6)
    best = rerank(query, cand, top_k=3)
    return generate(query, best)

def baseline_rag(query):
    return generate(query, vector_search_idx(query, 3))

q = "X-200 펌웨어 어떻게 올려요?"
print("[기준선]", baseline_rag(q))
print("\n[고급]  ", advanced_rag(q))


## 산출물 체크리스트
- ✅ BM25 + 벡터 **하이브리드(RRF)** 동작
- ✅ **Reranker** 2단계(N→K)
- ✅ HyDE·Multi-Query로 모호 질문 보강
- ✅ 기준선 대비 품질 비교 메모

> 💡 이 `advanced_rag()`가 Week 3에서 **에이전트의 검색 도구**가 됩니다.
